In [2]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)


# ==========================================
# 1. Load Dataset
# ==========================================

df = pd.read_csv(
    "data/market_data_final_features.csv",
    parse_dates=["Date"],
    index_col="Date"
)

print("Original Shape:", df.shape)


# ==========================================
# 2. Remove Initial HMM NaNs
# ==========================================

df = df.dropna(
    subset=["HMM_Regime_Risk"]
).copy()

print(
    "Shape After HMM Warmup Removal:",
    df.shape
)


# ==========================================
# 3. Target
# ==========================================

TARGET = "Crash_Risk"


# ==========================================
# 4. Remove Leakage / Non-Model Columns
# ==========================================

drop_columns = [

    TARGET,

    # Future information - STRICTLY REMOVE
    "Future_Min_SP500",
    "Future_Drawdown_10D",

    # String regime
    "Market_Regime",

    # Raw arbitrary HMM state
    "HMM_State",

    # Broken original PELT binary feature
    "PELT_Recent_Change"
]


X = df.drop(
    columns=drop_columns,
    errors="ignore"
)

y = df[TARGET]


# ==========================================
# 5. Remove Raw Price Levels
# ==========================================

# We prefer stationary returns/features instead
# of absolute market price levels.

raw_price_columns = [

    "SP500",
    "NASDAQ",
    "GOLD",
    "OIL",

    "SP500_MA20",
    "SP500_MA50"

]

X = X.drop(
    columns=raw_price_columns,
    errors="ignore"
)


print(
    "\nFeature Count:",
    X.shape[1]
)

print(
    "\nFeatures:"
)

for col in X.columns:
    print(col)


# ==========================================
# 6. Chronological Train / Test Split
# ==========================================

split_index = int(
    len(df) * 0.80
)

X_train = X.iloc[
    :split_index
]

X_test = X.iloc[
    split_index:
]

y_train = y.iloc[
    :split_index
]

y_test = y.iloc[
    split_index:
]


print(
    "\nTrain Period:",
    X_train.index.min(),
    "to",
    X_train.index.max()
)

print(
    "Test Period:",
    X_test.index.min(),
    "to",
    X_test.index.max()
)


print(
    "\nTrain Distribution:"
)

print(
    y_train.value_counts()
)


print(
    "\nTest Distribution:"
)

print(
    y_test.value_counts()
)


# ==========================================
# 7. Class Weight
# ==========================================

negative = (
    y_train == 0
).sum()

positive = (
    y_train == 1
).sum()

scale_pos_weight = (
    negative / positive
)

print(
    "\nScale Pos Weight:",
    scale_pos_weight
)


# ==========================================
# 8. CatBoost Model
# ==========================================

model = CatBoostClassifier(

    iterations=700,

    depth=6,

    learning_rate=0.03,

    loss_function="Logloss",

    eval_metric="AUC",

    scale_pos_weight=scale_pos_weight,

    random_seed=42,

    verbose=100

)


model.fit(

    X_train,
    y_train,

    eval_set=(
        X_test,
        y_test
    ),

    early_stopping_rounds=100,

    verbose=100
)


# ==========================================
# 9. Probability Prediction
# ==========================================

y_prob = model.predict_proba(
    X_test
)[:, 1]


# ==========================================
# 10. Default Threshold Evaluation
# ==========================================

y_pred_05 = (
    y_prob >= 0.50
).astype(int)


print(
    "\n===== THRESHOLD 0.50 ====="
)

print(
    classification_report(
        y_test,
        y_pred_05,
        digits=4
    )
)

print(
    "Confusion Matrix:"
)

print(
    confusion_matrix(
        y_test,
        y_pred_05
    )
)


# ==========================================
# 11. ROC-AUC + PR-AUC
# ==========================================

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

pr_auc = average_precision_score(
    y_test,
    y_prob
)


print(
    "\nROC-AUC:",
    round(
        roc_auc,
        4
    )
)

print(
    "PR-AUC:",
    round(
        pr_auc,
        4
    )
)


# ==========================================
# 12. Threshold Analysis
# ==========================================

precision, recall, thresholds = (
    precision_recall_curve(
        y_test,
        y_prob
    )
)

threshold_results = []

for threshold in np.arange(
    0.10,
    0.91,
    0.05
):

    pred = (
        y_prob >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_test,
        pred
    )

    tn, fp, fn, tp = cm.ravel()

    prec = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    rec = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * prec * rec /
        (prec + rec)
        if (prec + rec) > 0
        else 0
    )

    threshold_results.append(
        {
            "Threshold":
                round(
                    threshold,
                    2
                ),

            "Precision":
                round(
                    prec,
                    4
                ),

            "Recall":
                round(
                    rec,
                    4
                ),

            "F1":
                round(
                    f1,
                    4
                ),

            "False_Positive":
                fp,

            "False_Negative":
                fn
        }
    )


threshold_df = pd.DataFrame(
    threshold_results
)


print(
    "\n===== THRESHOLD ANALYSIS ====="
)

print(
    threshold_df.to_string(
        index=False
    )
)


# ==========================================
# 13. Feature Importance
# ==========================================

importance = pd.DataFrame({

    "Feature":
        X.columns,

    "Importance":
        model.get_feature_importance()

}).sort_values(

    "Importance",
    ascending=False

)


print(
    "\n===== TOP 20 FEATURES ====="
)

print(
    importance.head(
        20
    ).to_string(
        index=False
    )
)


# ==========================================
# 14. Save Outputs
# ==========================================

model.save_model(
    "catboost_crash_model.cbm"
)

importance.to_csv(
    "data/feature_importance.csv",
    index=False
)

threshold_df.to_csv(
    "data/threshold_analysis.csv",
    index=False
)


predictions = pd.DataFrame({

    "Actual":
        y_test,

    "Crash_Probability":
        y_prob,

    "Prediction_05":
        y_pred_05

})

predictions.to_csv(
    "data/test_predictions.csv"
)


print(
    "\nModel and results saved successfully."
)                           

Original Shape: (6423, 48)
Shape After HMM Warmup Removal: (5923, 48)

Feature Count: 36

Features:
VIX
SP500_Return
NASDAQ_Return
GOLD_Return
OIL_Return
SP500_Log_Return
NASDAQ_Log_Return
GOLD_Log_Return
OIL_Log_Return
SP500_Volatility_10
SP500_Volatility_20
NASDAQ_Volatility_20
SP500_Momentum_5
SP500_Momentum_10
SP500_Momentum_20
SP500_MA20_Distance
SP500_MA50_Distance
VIX_Change
VIX_MA10
VIX_Ratio
TDA_H0_Count
TDA_H0_TotalPersistence
TDA_H0_MaxPersistence
TDA_H0_MeanPersistence
TDA_H1_Count
TDA_H1_TotalPersistence
TDA_H1_MaxPersistence
TDA_H1_MeanPersistence
TDA_H1_Entropy
PELT_Days_Since_Change
PELT_Change_Strength
PELT_Change_30D
PELT_Change_60D
PELT_Recency_Score
PELT_Structural_Stress
HMM_Regime_Risk

Train Period: 2002-12-19 00:00:00 to 2021-10-14 00:00:00
Test Period: 2021-10-15 00:00:00 to 2026-07-02 00:00:00

Train Distribution:
Crash_Risk
0    4566
1     172
Name: count, dtype: int64

Test Distribution:
Crash_Risk
0    1144
1      41
Name: count, dtype: int64

Scale Pos Wei

In [3]:
# ==========================================
# SAVE COMPLETE EVALUATION REPORT
# ==========================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

report = classification_report(
    y_test,
    y_pred_05,
    digits=4
)

cm = confusion_matrix(
    y_test,
    y_pred_05
)

roc = roc_auc_score(
    y_test,
    y_prob
)

pr = average_precision_score(
    y_test,
    y_prob
)

with open(
    "data/model_evaluation.txt",
    "w"
) as f:

    f.write("CATBOOST CRASH PREDICTION RESULTS\n")
    f.write("=" * 50)

    f.write("\n\nTRAIN DISTRIBUTION\n")
    f.write(str(y_train.value_counts()))

    f.write("\n\nTEST DISTRIBUTION\n")
    f.write(str(y_test.value_counts()))

    f.write("\n\nCLASSIFICATION REPORT\n")
    f.write(report)

    f.write("\n\nCONFUSION MATRIX\n")
    f.write(str(cm))

    f.write(
        f"\n\nROC-AUC: {roc:.4f}"
    )

    f.write(
        f"\nPR-AUC: {pr:.4f}"
    )

    f.write(
        "\n\nTHRESHOLD ANALYSIS\n"
    )

    f.write(
        threshold_df.to_string(
            index=False
        )
    )

    f.write(
        "\n\nTOP 20 FEATURES\n"
    )

    f.write(
        importance.head(20)
        .to_string(index=False)
    )


print("\n================================")
print("FINAL MODEL RESULTS")
print("================================")

print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr, 4))

print("\nConfusion Matrix:")
print(cm)

print(
    "\nFull report saved:"
    " data/model_evaluation.txt"
)


FINAL MODEL RESULTS
ROC-AUC: 0.8468
PR-AUC: 0.1301

Confusion Matrix:
[[936 208]
 [ 14  27]]

Full report saved: data/model_evaluation.txt
